In [2]:
import sys

sys.path.append("..")

import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve

from src.metrics import evaluate_predictions

In [3]:
X_train = pd.read_csv("../data/processed/X_train_normal.csv")
X_val = pd.read_csv("../data/processed/X_val.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_val = pd.read_csv("../data/processed/y_val.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [4]:
IF_baseline = IsolationForest(
    n_estimators=100,
    max_samples="auto",
    contamination="auto",
    random_state=42
)

IF_baseline.fit(X_train)

val_scores = -IF_baseline.score_samples(X_val)
test_scores = -IF_baseline.score_samples(X_test)

val_metrics = evaluate_predictions(y_val, val_scores)
test_metrics = evaluate_predictions(y_test, test_scores)

print("Validation:")
print(val_metrics)

print("Test:")
print(test_metrics)

Validation:
{'precision': 0.09824561403508772, 'recall': 0.5656565656565656, 'f1': 0.16741405082212257, 'roc_auc': 0.9418325491518957, 'pr_auc': 0.14092872307148258}
Test:
{'precision': 0.11228070175438597, 'recall': 0.6530612244897959, 'f1': 0.19161676646706588, 'roc_auc': 0.9592925978776429, 'pr_auc': 0.18974450546758204}


In [7]:
def best_threshold_by_f1(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)

    best_index = np.argmax(f1_scores[:-1])
    best_threshold = thresholds[best_index]

    return best_threshold

In [10]:
from sklearn.model_selection import ParameterGrid

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_samples": ["auto", 512, 1024],
    "contamination": ["auto", 0.001, 0.002, 0.005],
    "max_features": [0.7, 1.0]
}

results = []

best_f1 = 0
best_result = None

for params in ParameterGrid(param_grid):

    IF_model = IsolationForest(
        n_estimators=params["n_estimators"],
        max_samples=params["max_samples"],
        contamination=params["contamination"],
        max_features=params["max_features"],
        random_state=42
    )

    IF_model.fit(X_train)

    val_scores = -IF_model.score_samples(X_val)

    threshold = best_threshold_by_f1(y_val, val_scores)

    metrics = evaluate_predictions(
        y_true=y_val,
        scores=val_scores,
        threshold=threshold
    )

    result = {
        "n_estimators": params["n_estimators"],
        "max_samples": params["max_samples"],
        "contamination": params["contamination"],
        "max_features": params["max_features"],
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"]
    }

    results.append(result)

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_result = result

print("BEST RESULT:")
print(best_result)

BEST RESULT:
{'n_estimators': 100, 'max_samples': 1024, 'contamination': 'auto', 'max_features': 1.0, 'threshold': np.float64(0.6497942179673687), 'precision': 0.26666666666666666, 'recall': 0.2828282828282828, 'f1': 0.27450980392156865, 'roc_auc': 0.9410017378292004, 'pr_auc': 0.13558911749494962}


In [11]:
results_df = pd.DataFrame(results)

top10 = results_df.sort_values(by="f1", ascending=False).head(10)

top10

,n_estimators,max_samples,contamination,max_features,threshold,precision,recall,f1,roc_auc,pr_auc
15,100,1024,auto,1.0,0.649794,0.266667,0.282828,0.274510,0.941002,0.135589
51,100,1024,0.002,1.0,0.649794,0.266667,0.282828,0.274510,0.941002,0.135589
33,100,1024,0.001,1.0,0.649794,0.266667,0.282828,0.274510,0.941002,0.135589
69,100,1024,0.005,1.0,0.649794,0.266667,0.282828,0.274510,0.941002,0.135589
63,100,auto,0.005,1.0,0.629657,0.228188,0.343434,0.274194,0.941833,0.140929
45,100,auto,0.002,1.0,0.629657,0.228188,0.343434,0.274194,0.941833,0.140929
9,100,auto,auto,1.0,0.629657,0.228188,0.343434,0.274194,0.941833,0.140929
27,100,auto,0.001,1.0,0.629657,0.228188,0.343434,0.274194,0.941833,0.140929
52,200,1024,0.002,1.0,0.645085,0.237288,0.282828,0.258065,0.942904,0.130899
34,200,1024,0.001,1.0,0.645085,0.237288,0.282828,0.258065,0.942904,0.130899


In [12]:
IF_best = IsolationForest(
    n_estimators=100,
    max_samples=1024,
    contamination="auto",
    max_features=1.0,
    random_state=42
)

IF_best.fit(X_train)

test_scores = -IF_best.score_samples(X_test)

test_metrics = evaluate_predictions(
    y_true=y_test,
    scores=test_scores,
    threshold=0.6497942179673687
)

print(test_metrics)

{'precision': 0.28205128205128205, 'recall': 0.336734693877551, 'f1': 0.30697674418604654, 'roc_auc': 0.9553223301138125, 'pr_auc': 0.19073645265641728}
